In [0]:
%fs 
ls /mnt/ecomsalesdbb/customer/

path,name,size,modificationTime
dbfs:/mnt/ecomsalesdbb/customer/customer_delta.parquet,customer_delta.parquet,1479,1725289991000
dbfs:/mnt/ecomsalesdbb/customer/customers.parquet,customers.parquet,4497046,1725282986000


In [0]:
#load history data

from pyspark.sql.functions import lit,now,to_date,xxhash64,col

customer= spark.read.parquet("dbfs:/mnt/ecomsalesdbb/customer/customers.parquet")


customer1=customer.withColumn("Active_Flag",lit("Y")).withColumn("From_Date",to_date(now()))\
    .withColumn("To_Date",to_date(lit('9999-12-31')))

display(customer1)

customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date
861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Y,2024-09-02,9999-12-31
290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Y,2024-09-02,9999-12-31
060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Y,2024-09-02,9999-12-31
259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Y,2024-09-02,9999-12-31
345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Y,2024-09-02,9999-12-31
4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,Y,2024-09-02,9999-12-31
addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,Y,2024-09-02,9999-12-31
57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,Y,2024-09-02,9999-12-31
1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,Y,2024-09-02,9999-12-31
9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,Y,2024-09-02,9999-12-31


In [0]:
spark.sql("create database if not exists customer_master;")
customer1.write.mode("overwrite").saveAsTable("customer_master.customers_hist")


In [0]:
%sql
select xxhash64(customer_d1.customer_zip_code_prefix,customer_d1.customer_city,customer_d1.customer_state  ) ,
xxhash64('14410',customer_d1.customer_city,customer_d1.customer_state  ) ,
* from customer_master.customers_hist customer_d1
where customer_id='861eff4711a542e4b93843c6dd7febb0'

"xxhash64(customer_zip_code_prefix, customer_city, customer_state)","xxhash64(14410, customer_city, customer_state)",customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date
4856160795173122497,-27257565422297328,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Y,2024-09-02,9999-12-31


In [0]:
customer_hist=spark.read.table("customer_master.customers_hist")
#read delta table
customer_d= spark.read.parquet("dbfs:/mnt/ecomsalesdbb/customer/customer_delta.parquet")

customer_d1=customer_d.withColumn("Active_Flag",lit("Y")).withColumn("From_Date",to_date(now()))\
    .withColumn("To_Date",to_date(lit('9999-12-31')))


customer_d1.display()

customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date
861eff4711a542e4b93843c6dd7febb0,15000,franca,SP,Y,2024-09-02,9999-12-31
290c77bc529b7ac935b93aa66c333dc3,9790,sao paulo,SP,Y,2024-09-02,9999-12-31
9999999999999999999999999990,81560,timoteo,MG,Y,2024-09-02,9999-12-31


In [0]:
#scd type 2

update_ds=customer_hist.join(customer_d1,((customer_hist.customer_id==customer_d1.customer_id) & (customer_hist.Active_Flag =='Y') ),"inner")\
    .filter(xxhash64(customer_hist.customer_zip_code_prefix,customer_hist.customer_city,customer_hist.customer_state ) !=
            xxhash64(customer_d1.customer_zip_code_prefix,customer_d1.customer_city,customer_d1.customer_state  ) )\
                .select(customer_hist.customer_id,
                        customer_hist.customer_zip_code_prefix,
                        customer_hist.customer_city,
                        customer_hist.customer_state,
                        lit("N").alias("Active_Flag"),
                        customer_hist.From_Date,
                        lit(to_date(now())).alias("To_Date"))
update_ds.display()

customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date
861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,N,2024-09-02,2024-09-02
290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,N,2024-09-02,2024-09-02


In [0]:
no_change=customer_hist.join(update_ds,((customer_hist.customer_id==update_ds.customer_id) & (customer_hist.Active_Flag =='Y') ),"left_anti")
insert_ds = customer_d1.join(no_change,"customer_id","left_anti")

consolidate=no_change.union(update_ds).union(insert_ds)

 

consolidate.write.mode("overwrite").saveAsTable("customer_master.customers_hist")


In [0]:
%sql
select * from customer_master.customers_hist 
 where customer_id in('861eff4711a542e4b93843c6dd7febb0','290c77bc529b7ac935b93aa66c333dc3','9999999999999999999999999990')
order by customer_id

customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date
290c77bc529b7ac935b93aa66c333dc3,9790,sao paulo,SP,Y,2024-09-02,9999-12-31
290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,N,2024-09-02,2024-09-02
861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,N,2024-09-02,2024-09-02
861eff4711a542e4b93843c6dd7febb0,15000,franca,SP,Y,2024-09-02,9999-12-31
9999999999999999999999999990,81560,timoteo,MG,Y,2024-09-02,9999-12-31


In [0]:
from delta.tables import *
cust_tbl_inst = DeltaTable.forPath(spark, 'dbfs:/user/hive/warehouse/customer_master.db/customers_hist')

#(cust_tbl_inst.toDF()).display()

In [0]:

#scd2 with MERGE

update_ds=customer_hist.join(customer_d1,(customer_hist.customer_id==customer_d1.customer_id) & (customer_hist.Active_Flag=='Y'),"inner")\
    .filter(xxhash64(customer_hist.customer_zip_code_prefix,customer_hist.customer_city,customer_hist.customer_state ) !=
            xxhash64(customer_d1.customer_zip_code_prefix,customer_d1.customer_city,customer_d1.customer_state  ) )\
                .select(customer_d1.customer_id.alias("merge_key"),customer_d1['*'])


In [0]:

insert_ds=customer_d1.join(customer_hist,(customer_d1.customer_id==customer_hist.customer_id) & (customer_hist.Active_Flag=='Y'),"left_outer")\
    .filter(xxhash64(customer_hist.customer_zip_code_prefix,customer_hist.customer_city,customer_hist.customer_state ) !=
            xxhash64(customer_d1.customer_zip_code_prefix,customer_d1.customer_city,customer_d1.customer_state  ))\
    .select(lit(None).alias("merge_key"),customer_d1['*']) 

In [0]:
consolidate=update_ds.unionByName(insert_ds)


display(consolidate)

merge_key,customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date


In [0]:
cust_tbl_inst.alias("trg").merge(consolidate.alias("src"),condition="trg.customer_id=src.merge_key and trg.Active_Flag='Y'"

                       ).whenMatchedUpdate(set ={"Active_Flag":"'N'","To_Date":"to_date(now())"}

                                           ).whenNotMatchedInsert(values={

"customer_id": "customer_id",
"customer_zip_code_prefix": "customer_zip_code_prefix",
"customer_city": "customer_city",
"customer_state": "customer_state",
"Active_Flag": "'Y'",
"From_Date": "to_date(now())",
"To_Date": "'9999-12-31'"}).execute()

In [0]:
%sql
select * from customer_master.customers_hist where customer_id in('861eff4711a542e4b93843c6dd7febb0','290c77bc529b7ac935b93aa66c333dc3','9999999999999999999999999990') 
order by customer_id

customer_id,customer_zip_code_prefix,customer_city,customer_state,Active_Flag,From_Date,To_Date
290c77bc529b7ac935b93aa66c333dc3,9790,sao paulo,SP,Y,2024-09-02,9999-12-31
290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,N,2024-09-02,2024-09-02
861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,N,2024-09-02,2024-09-02
861eff4711a542e4b93843c6dd7febb0,15000,franca,SP,Y,2024-09-02,9999-12-31
9999999999999999999999999990,81560,timoteo,MG,Y,2024-09-02,9999-12-31
